# Creating the vector store

## There are several ways of creating the configuration
_INDEX_NAME = "my_test_document_storage"


def _get_embedding_dim(model) -> int:
    """
    Probe qwen3_embedding_model once and cache the resulting vector
    length. Not hardcoded, since it depends on which qwen3-embedding
    tag (0.6b/4b/8b) is pulled locally — each has a different output
    dimension.
    """
    global _embedding_dim
    if _embedding_dim is None:
        _embedding_dim = len(model.embed_query("dimension probe"))
    return _embedding_dim

def _build_redis_store() -> RedisVectorStore:
    embeddings = qwen3_embedding_model
    config = RedisConfig(
        index_name=_INDEX_NAME,
        redis_url=os.getenv("REDIS_URL", "redis://localhost:6379"),
        indexing_algorithm="HNSW",
        embedding_dimensions=_get_embedding_dim(embeddings),
        from_existing=False,
    )

    return RedisVectorStore(embeddings=embeddings, config=config)

In [6]:

from langchain_redis import RedisVectorStore
import models.embedding_models.ollama_models

vector_store = RedisVectorStore(
    models.embedding_models.ollama_models.qwen3_embedding_model,
    redis_url="redis://localhost:6379",
    index_name="my_documents"
)

[-0.016565695, -0.06390685, -0.015351882, -0.038346153, 0.067234874, 0.07391147, 0.019366918, -0.028430518, -0.055937797, 0.030653445, 0.026487209, -0.011634193, 0.018879348, -0.014205319, -0.05616461, 0.12932058, 0.023646282, 0.06322779, 0.017105944, -0.03029193, -0.05147567, 0.05669777, -0.05461983, 0.08540411, -0.008365315, 0.043155782, -0.06685639, 0.041184828, -0.03973803, -0.043367796, 0.04714078, 0.03665677, -0.061637748, -0.03912709, -0.021573499, -0.01883078, -0.0068341712, -0.015950875, -0.05809134, 0.06832146, -0.006836803, 0.015750468, -0.032709047, 0.009048875, 0.017173788, -0.0013285076, 0.0054164035, 0.030433292, 0.035834428, -0.005513822, -0.051199093, -0.03633166, 0.0017785345, -0.025859121, -0.0021392414, -0.042527657, 0.03323818, -0.0171316, 0.0064062597, 0.014172643, -0.026899166, 0.106962055, -0.05209595, 0.06021733, 0.016102143, 0.026442707, 0.0024704228, -0.053226754, -0.009961247, -0.017317204, -0.003817884, -0.013603202, -0.0050076, 0.045111727, -0.004757223, 0

## To fecth the dimension of the embeddng model
### this is actually a indirect way, just get a embedding of the query and get the lenght of it

In [7]:
embedding_model =models.embedding_models.ollama_models.qwen3_embedding_model
print(embedding_model.embed_query("dimension probe"))


[-0.016564185, -0.06390394, -0.015351705, -0.038349558, 0.06724141, 0.073906176, 0.019362593, -0.028439673, -0.055940874, 0.030657364, 0.026480649, -0.011632324, 0.018876193, -0.014205241, -0.05616283, 0.12931654, 0.023649154, 0.06322228, 0.017099375, -0.030291893, -0.05147899, 0.05669248, -0.054624334, 0.08541135, -0.008363293, 0.0431526, -0.06684815, 0.04118516, -0.039738286, -0.043368477, 0.047123503, 0.036657397, -0.061637342, -0.039132513, -0.02157365, -0.018830642, -0.006836449, -0.015949138, -0.058093525, 0.068327926, -0.006835584, 0.015747953, -0.032714445, 0.009050019, 0.017178481, -0.0013232767, 0.005410983, 0.030441722, 0.035830397, -0.0055125034, -0.0511958, -0.03633474, 0.0017740336, -0.025859842, -0.0021430105, -0.04253366, 0.033242725, -0.017126603, 0.006408812, 0.014171264, -0.026901156, 0.10695143, -0.052095033, 0.060218703, 0.016104395, 0.026443414, 0.002473778, -0.05322081, -0.00995949, -0.017317874, -0.003831747, -0.01360098, -0.0050033377, 0.04511245, -0.004754493,

# Storing then embedding documents

In [3]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content="Java supports virtual threads.",
        metadata={"topic": "java"}
    ),
    Document(
        page_content="Python supports list comprehensions.",
        metadata={"topic": "python"}
    ),
    Document(
        page_content="LangGraph is used for building stateful agent workflows.",
        metadata={"topic": "langgraph"}
    ),
    Document(
        page_content="LangChain provides abstractions for working with LLMs.",
        metadata={"topic": "langchain"}
    )
]
vector_store.add_documents(documents)

['my_documents:01M33TXW8KT0N74TWTEQT7JGJJ',
 'my_documents:01M33TXW8KT0N74TWTEQT7JGJK',
 'my_documents:01M33TXW8KT0N74TWTEQT7JGJM',
 'my_documents:01M33TXW8KT0N74TWTEQT7JGJN']

# Retrieve the documents using similarity search

In [4]:
question = "What is LangGraph?"
results = vector_store.similarity_search(
    question,k=2
)

for doc in results:
    print(doc)

page_content='LangGraph is used for building stateful agent workflows.' metadata={'topic': 'langgraph'}
page_content='LangChain provides abstractions for working with LLMs.' metadata={'topic': 'langchain'}


# Retrieve the documents using retriever

In [5]:
retriever = vector_store.as_retriever(
    search_kwargs={"k": 3}
)


docs = retriever.invoke(question)
for doc in results:
    print(doc)

page_content='LangGraph is used for building stateful agent workflows.' metadata={'topic': 'langgraph'}
page_content='LangChain provides abstractions for working with LLMs.' metadata={'topic': 'langchain'}


# To check the document in the  redis
## redis-cli KEYS '*'
## redis-cli SCAN 0

# Comparison between InMemory and Redis Vector store

## InMemoryVectorStore vs RedisVectorStore

| Feature | InMemoryVectorStore | RedisVectorStore |
|---|---|---|
| Storage | Python process memory | Redis |
| Persistence | ❌ No | ✅ Yes |
| Survives application restart | ❌ No | ✅ Yes |
| Shared across application instances | ❌ No | ✅ Yes |
| External infrastructure required | ❌ No | ✅ Yes |
| Vector similarity search | ✅ Yes | ✅ Yes |
| Metadata support | ✅ Yes | ✅ Yes |
| Retriever support | ✅ Yes | ✅ Yes |
| Configurable `k` | ✅ Yes | ✅ Yes |
| Filtering | Limited | ✅ Richer options |
| Scalability | Limited by application memory | Much better |
| Suitable for unit tests | ✅ Excellent | ⚠️ Usually unnecessary |
| Suitable for prototypes | ✅ Excellent | ✅ Yes |
| Suitable for production | ⚠️ Limited | ✅ Yes |
| Data shared between processes | ❌ No | ✅ Yes |
| Operational complexity | Very low | Higher |
| Speed for small datasets | Very fast | Network overhead |
| Best use case | Learning, testing, small RAG | Production/shared RAG |